# Fase 2 — Fundação (Marco 2: T-2.1 a T-2.8)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PabloThePmX/dengue-comorbidity-risk/blob/feat/lora/notebooks/02_fundacao.ipynb)

Notebook de execução e verificação no Google Colab (NFR-05, D-56). Clona automaticamente a branch `feat/lora`, instala o ambiente e grava os relatórios em `MyDrive/dengue-comorbidity-risk/reports/verificacao/`.


In [ ]:
# CÉLULA: setup-colab (Criação automática do ambiente e projeto no Colab)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/PabloThePmX/dengue-comorbidity-risk.git"
BRANCH = "feat/lora"
REPO_DIR = Path("/content/dengue-comorbidity-risk")

if "google.colab" in sys.modules:
    if not (REPO_DIR / ".git").exists():
        subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

from dengue_sinan.colab import prepare_colab_project, sync_and_package_reports  # noqa: E402

cfg = prepare_colab_project(anos_necessarios=(2022, 2023))


In [ ]:
# CÉLULA: T-2.1 — Carregar a população em memória e gerar relatório de verificação
from dengue_sinan.dataset import build_feature_frame, load_dev_population, verify_t2_1

relatorio_t2_1 = verify_t2_1(cfg)
print(relatorio_t2_1.read_text(encoding="utf-8"))

pop_dev = load_dev_population(cfg)
df_dev = build_feature_frame(pop_dev)


In [ ]:
# CÉLULA: T-2.3 — Dobras leave-one-region-out e validação interna (com hash)
from dengue_sinan.folds import verify_t2_3

relatorio_t2_3 = verify_t2_3(cfg, df_dev)
print(relatorio_t2_3.read_text(encoding="utf-8"))


In [ ]:
# CÉLULA: T-2.6 e T-2.7 — Análise descritiva e eventos por par/combinação de comorbidades
from dengue_sinan.descriptive import verify_t2_6_t2_7

relatorio_t2_6, relatorio_t2_7 = verify_t2_6_t2_7(cfg, df_dev)
print(relatorio_t2_6.read_text(encoding="utf-8"))
print(relatorio_t2_7.read_text(encoding="utf-8"))


In [ ]:
# CÉLULA: T-2.8 — Baselines B0, B1 e B2
from dengue_sinan.baselines import verify_t2_8

relatorio_t2_8 = verify_t2_8(cfg, df_dev)
print(relatorio_t2_8.read_text(encoding="utf-8"))

zip_path = sync_and_package_reports(cfg, zip_path="reports_fase2_verificacao.zip")
if "google.colab" in sys.modules:
    from google.colab import files

    files.download(str(zip_path))
